<a href="https://colab.research.google.com/github/HarshiniArulmani2006/Harshini-codeboosters-2026/blob/main/Day_8/Day_8_RAG_mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install sentence-transformers chromadb groq pandas -q
print('Installation Completed!!!')

Installation Completed!!!


In [11]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print(f'All Libraries Successfully!!!!')

All Libraries Successfully!!!!


In [12]:
GROQ_API_KEY="xxxx"
os.environ["GROQ_API_KEY"]=GROQ_API_KEY #os.environ acts like locker key,when someone see our project it doesn't show our api key
groq_client=Groq(api_key=GROQ_API_KEY)
print('Groq API client initialised')
print('Note:If you see an authentication error later,double check your API Key')

Groq API client initialised
Note:If you see an authentication error later,double check your API Key


In [13]:
df=pd.read_csv('college_notes.csv')
print("Shape of Data Set:",df.shape)
print("\nColumn Names:",df.columns.tolist())
print("\nFirst 5 Rows:")
print(df.head(5).to_string(index=False))
print(df.columns)

Shape of Data Set: (15, 4)

Column Names: ['note_id', 'subject', 'topic', 'content']

First 5 Rows:
note_id          subject                    topic                                                                                                                                                                                                                                            content
   N001 Data Engineering            ETL Pipelines                           ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.
   N002 Data Engineering            SQL Databases                                  A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interact with relational databases. Common SQL commands include SELECT INSERT UPDATE and DELETE.
   N003 Data Engineer

In [14]:
documents=df['content'].to_list()
ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]
metadata=[
    {'subject':row['subject'],'topic':row['topic']}
    for row in df.to_dict('records')
]
print(f"Total chunks prepared:{len(documents)}")
print(f"First document ID:{ids[2]}")
print(f"First metadata:{metadata[2]}")
print(f"First document:{documents[2][:100]}...")

Total chunks prepared:15
First document ID:note_N003
First metadata:{'subject': 'Data Engineering', 'topic': 'Data Cleaning'}
First document:Data cleaning involves fixing or removing incorrect incomplete duplicate or corrupted data. Common c...


In [15]:
#convert to chunks
documents=df['content'].tolist()
ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas=[
    {"subject":row['subject'],"topic":row['topic']}
    for row in df.to_dict('records')
]
print(f"Total Chunks Prepared:{len(documents)}")
print(f"First document's ID:{ids[3]}")
print(f"First document's Metadata:{metadatas[3]}")
print(f"First 100 characters of document:{documents[3][:100]}....")


Total Chunks Prepared:15
First document's ID:note_N004
First document's Metadata:{'subject': 'Data Engineering', 'topic': 'APIs and Data Collection'}
First 100 characters of document:An API or Application Programming Interface allows two software applications to talk to each other. ....


In [16]:
print("Loading Embedding Model..........")
embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding Model Loaded successfully!!!!')
test_embedding=embedding_model.encode("This is the Test Sentence")
print(f'Test Embedding Shape:{test_embedding.shape}')
print(f"First 5 values of test embedding:{test_embedding[:5]}")

Loading Embedding Model..........


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Model Loaded successfully!!!!
Test Embedding Shape:(384,)
First 5 values of test embedding:[ 0.07163641  0.07820371 -0.0090355   0.08910967  0.03642739]


In [17]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection(name="new_college_notes_rag")
print("ChromaDB client created.")
print(f"Collection name:new_college_notes_rag")
print(f"Documents in collection so far:{collection.count()}")

ChromaDB client created.
Collection name:new_college_notes_rag
Documents in collection so far:0


In [18]:
print("Generating Embeddings for all 15 notes")
embeddings=embedding_model.encode(documents,show_progress_bar=True) #show_progress_bar=True-->used for the green bar in output(optional)
print(f"\nEmbedding matrix Shape:{embeddings.shape}")
embeddings_list=embeddings.tolist()
collection.add(
    documents=documents,
    embeddings=embeddings_list,
    ids=ids,
    metadatas=metadatas
)
print(f"\nDocuments Added Successfully in ChromaDB!!!!")
print(f"Total documents in collection:{collection.count()}")

Generating Embeddings for all 15 notes


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix Shape:(15, 384)

Documents Added Successfully in ChromaDB!!!!
Total documents in collection:15


In [19]:
#creating the chunks and retrieve
def retrieve_relevant_chunks(question,top_k=3):
  """
  Given a user question,retrieve the most relevant document chunks from ChromaDB
  Parameters:
  question(str):The user's question as a text string
  top_k(int):How many top results to return(default:3)
  Returns:
  A dictionary containing retrieved documents,distances and metadata
  """
  question_embedding=embedding_model.encode(question).tolist()
  results=collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k,
  )
  return results
print("Retrieval Function defined successfully")
print("Function:retrieve_relevant_chunks(question,top_k=3)")

Retrieval Function defined successfully
Function:retrieve_relevant_chunks(question,top_k=3)


In [20]:
test_question="What is ETL and how does it work in data engineering"
print(f"Test Question:{test_question}")
results=retrieve_relevant_chunks(test_question,top_k=3)
print("\nTop 3 Retrieved Chunks:")
for i,(doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
    )):
  print(f"\nResults{i+1}:")
  print(f"-->Subject:{meta['subject']}")
  print(f"-->Topic:{meta['topic']}")
  print(f"-->Distance:{dist:.4f}")
  print(f"-->Content:{doc[:120]}.....")

Test Question:What is ETL and how does it work in data engineering

Top 3 Retrieved Chunks:

Results1:
-->Subject:Data Engineering
-->Topic:ETL Pipelines
-->Distance:0.2041
-->Content:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i.....

Results2:
-->Subject:Data Engineering
-->Topic:APIs and Data Collection
-->Distance:1.1100
-->Content:An API or Application Programming Interface allows two software applications to talk to each other. In data engineering .....

Results3:
-->Subject:Python Programming
-->Topic:Data Visualization
-->Distance:1.3892
-->Content:Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo.....


In [21]:
def build_context_from_results(results):
  """
  Format ChromaDB retrieval results into a readable context string.
  Parameters:
  results:The output from collection.query()-a dictionary
  Returns:
  context_str(str):A formatted string od all retrieved document chunks
  """
  context_parts=[] #empty list to collect formatted chunks
  for i,(doc,meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
    chunk_text=f"[Source{i+1}:{meta['subject']}-{meta['topic']}]\n{doc}"
    context_parts.append(chunk_text)
  return "\n\n".join(context_parts)

In [23]:
question=input("Enter your question: ")
# Retrieve top 3 relevant chunks
results=retrieve_relevant_chunks(question,top_k=3)
print("TOP 3 RETRIEVED NOTES")
for i,(doc, meta, dist) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
    print(f"\nResult {i+1}")
    print(f"Subject:{meta['subject']}")
    print(f"Topic:{meta['topic']}")
    print(f"Distance:{dist:.4f}")
    print(f"Content:\n{doc}")
context=build_context_from_results(results)
print("\n")
print("CONTEXT SENT TO LLM")
print(context)

Enter your question: what is ETL
TOP 3 RETRIEVED NOTES

Result 1
Subject:Data Engineering
Topic:ETL Pipelines
Distance:0.3395
Content:
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.

Result 2
Subject:Data Engineering
Topic:APIs and Data Collection
Distance:1.5858
Content:
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.

Result 3
Subject:Generative AI
Topic:Retrieval Augmented Generation
Distance:1.6215
Content:
RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This reduces hallucination and allows AI to a